<a href="https://colab.research.google.com/github/EMADUDDINAsdaq/federated-learning-fairness-xray/blob/main/gifair_per_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federated Learning — Method 5: GIFAIR-FL-Per (Yue et al. 2022)

**Algorithm 3 — Yue et al. 2022 (exact):**
Personalised variant of GIFAIR-Global that addresses its main limitation.
GIFAIR-Global needs an extra communication round to compute r_k from global
model losses. GIFAIR-Per circumvents this by computing r_k from each client's
own personalised model losses — no extra round needed.

Key differences from GIFAIR-Global (Algorithm 2):
- r_k computed from PERSONALISED losses F_k(theta_k), not global model losses
- Each client retains its own theta_k after each round (personalised solution)
- Server stores one set of personalised weights per hospital
- Final evaluation uses each hospital's theta_k, not the global model
- Prevents Hospital_B extreme distribution from destabilising global model

Citation: Yue et al. 2022, INFORMS Journal on Data Science.
lambda=0.1

## Section 1 — Environment Setup

In [ ]:
pip install flwr protobuf

In [ ]:
!pip install "flwr[simulation]" protobuf -q
print("✓ Libraries installed")

✓ Libraries installed


In [ ]:
import flwr as fl
print(f"flwr : {fl.__version__}")
print("✓ Flower working")

flwr : 1.32.1
✓ Flower working


In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
import flwr as fl
from flwr.common import (NDArrays, Scalar, Parameters,
                          parameters_to_ndarrays, ndarrays_to_parameters,
                          FitIns, FitRes, EvaluateIns, EvaluateRes)
from flwr.server.strategy import Strategy
from flwr.server.client_proxy import ClientProxy
from typing import Dict, List, Optional, Tuple
warnings.filterwarnings('ignore')

print(f"flwr  : {fl.__version__}")
print(f"torch : {torch.__version__}")
print(f"numpy : {np.__version__}")
print(f"GPU   : {torch.cuda.is_available()}")

flwr  : 1.32.1
torch : 2.11.0+cu128
numpy : 2.0.2
GPU   : True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import flwr.simulation
print("Simulation module loaded")

Simulation module loaded


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=== Session Initialisation ===")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"CUDA   : {torch.version.cuda}")
    print("\n✓ GPU ready")
else:
    print("\n⚠ No GPU — Runtime → Change runtime type → A100")

=== Session Initialisation ===
Device : cuda
GPU    : NVIDIA L4
Memory : 23.7 GB
CUDA   : 12.8

✓ GPU ready


## Section 2 — Dataset Download and Image Indexing

In [ ]:
import shutil

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/dissertation/kaggle.json',
            '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✓ Kaggle credentials loaded")

os.system('pip install -q kaggle')
os.system('kaggle datasets download -d nih-chest-xrays/data '
          '--path /content/nih_kaggle --unzip --quiet')

DATASET_PATH = '/content/nih_kaggle'
print(f"✓ Dataset path: {DATASET_PATH}")

✓ Kaggle credentials loaded
✓ Dataset path: /content/nih_kaggle


## Section 3 — Load Frozen Splits

In [ ]:
SPLIT_DIR = '/content/drive/MyDrive/dissertation/splits'
HOSPITAL_NAMES = ['Hospital_A', 'Hospital_B', 'Hospital_C',
                  'Hospital_D', 'Hospital_E']
NUM_CLIENTS = 5

train_clients = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_train.csv') for n in HOSPITAL_NAMES}
val_clients   = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_val.csv')   for n in HOSPITAL_NAMES}
test_clients  = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_test.csv')  for n in HOSPITAL_NAMES}
#Hospital_A_test.csv
for name in HOSPITAL_NAMES:
    print(f"{name}: {len(train_clients[name]):,} train / "
          f"{len(val_clients[name]):,} val / {len(test_clients[name]):,} test")

sample_path = train_clients['Hospital_A']['image_path'].iloc[0]
assert os.path.exists(sample_path), f"Path not found: {sample_path} — check Kaggle download completed"
print("✓ Image paths resolve correctly in this session")

Hospital_A: 52,332 train / 6,168 val / 3,005 test
Hospital_B: 9,074 train / 1,073 val / 508 test
Hospital_C: 29,813 train / 3,412 val / 1,711 test
Hospital_D: 1,276 train / 145 val / 74 test
Hospital_E: 2,997 train / 348 val / 168 test
✓ Image paths resolve correctly in this session


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 4 — GPU Optimisation

In [ ]:
torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False

BATCH_SIZE  = 512
NUM_WORKERS = 4
PREFETCH    = 2

print(f"✓ BATCH_SIZE  : {BATCH_SIZE}")
print(f"✓ NUM_WORKERS : {NUM_WORKERS}")
print(f"✓ PREFETCH    : {PREFETCH}")

✓ BATCH_SIZE  : 512
✓ NUM_WORKERS : 4
✓ PREFETCH    : 2


## Section 5 — Transforms, Dataset

In [ ]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

print(f"✓ Image size    : {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"✓ Normalisation : ImageNet mean/std")

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return image, label

print("✓ ChestXrayDataset defined")

✓ Image size    : 224×224
✓ Normalisation : ImageNet mean/std
✓ ChestXrayDataset defined


## Section 6 —  Model

In [ ]:
ROUNDS = 10

def build_model():
    model    = models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

test_model = build_model()
params     = sum(p.numel() for p in test_model.parameters())
print(f"✓ ResNet-18 — ImageNet pretrained")
print(f"✓ Parameters  : {params:,}")
print(f"✓ Rounds      : {ROUNDS}")
del test_model

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 240MB/s]


✓ ResNet-18 — ImageNet pretrained
✓ Parameters  : 11,177,025
✓ Rounds      : 10


## Section 7 — Evaluation Function

In [ ]:
def evaluate_client(model, dataframe, device):
    model = model.to(device)
    model.eval()

    loader = DataLoader(
        ChestXrayDataset(dataframe, transform=val_transform),
        batch_size         = BATCH_SIZE,
        shuffle            = False,
        num_workers        = NUM_WORKERS,
        pin_memory         = True,
        persistent_workers = True
    )

    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            probs = torch.sigmoid(
                model(images.to(device, non_blocking=True))
            ).cpu().numpy()
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs > 0.5).astype(int)

    def auc_fnr_for_mask(y_true, y_prob, y_pred):
        if len(y_true) < 10 or y_true.sum() == 0:
            return float('nan'), float('nan')
        try:
            auc = float(roc_auc_score(y_true, y_prob))
        except Exception:
            auc = float('nan')
        fn  = int(((y_pred == 0) & (y_true == 1)).sum())
        tp  = int(((y_pred == 1) & (y_true == 1)).sum())
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        return round(auc, 4), round(fnr, 4)

    auc, fnr = auc_fnr_for_mask(labels, probs, preds)
    acc      = round(float((preds == labels).mean() * 100), 2)
    metrics  = {'auc': auc, 'fnr': fnr, 'accuracy': acc}

    for sex in ['M', 'F']:
        mask = dataframe['Patient Sex'].values == sex
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{sex}'] = a
            metrics[f'fnr_{sex}'] = f

    for grp in ['0-20', '20-40', '40-60', '60-80', '80+']:
        mask = dataframe['Age Group'].values == grp
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{grp}'] = a
            metrics[f'fnr_{grp}'] = f

    return metrics

print("✓ evaluate_client() defined")
print("  Metrics : AUC + FNR (overall, per sex, per age group)")

✓ evaluate_client() defined
  Metrics : AUC + FNR (overall, per sex, per age group)


## Section 8 — GIFAIR-FL-Per (Yue et al. 2022)

**Algorithm 3 — Yue et al. 2022:**  
Difference from GIFAIR-Global:  
- r_k computed from PERSONALISED losses {theta_k}, not global model  
- Each client retains its own theta_k after each round  
- Server stores one set of personalised weights per hospital  
- Evaluation uses personalised models, not global model  
This prevents extreme clients from destabilising the global model.

In [ ]:
# GIFAIR-Per client — identical to GIFAIR-Global client
# Gradient scaling inside local training is the same
# The difference is handled entirely in the strategy

LAM = 0.1

class GIFAIRPerHospitalClient(fl.client.NumPyClient):

    def __init__(self, name: str, dataframe, val_dataframe, device):
        self.name          = name
        self.dataframe     = dataframe
        self.val_dataframe = val_dataframe
        self.device        = device
        self.model         = build_model().to(device)

    def get_parameters(self, config) -> NDArrays:
        return [v.cpu().numpy() for v in self.model.state_dict().values()]

    def set_parameters(self, parameters: NDArrays):
        state_dict = dict(zip(
            self.model.state_dict().keys(),
            [torch.tensor(p) for p in parameters]
        ))
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters: NDArrays, config: Dict) -> Tuple[NDArrays, int, Dict]:
        self.set_parameters(parameters)
        self.model.train()

        epochs = int(config.get('epochs', 3))
        lr     = float(config.get('lr', 1e-4))
        scale  = float(config.get('scale', 1.0))

        loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        criterion = nn.BCEWithLogitsLoss()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)

        total_loss, total_samples = 0.0, 0
        for _ in range(epochs):
            for images, labels in loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True).unsqueeze(1)

                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()

                # Yue et al. 2022 Algorithm 3 — scale gradients
                for param in self.model.parameters():
                    if param.grad is not None:
                        param.grad.data.mul_(scale)

                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples

        # Also compute personalised loss for r_k update [Yue et al. 2022 Eq.5]
        self.model.eval()
        per_loss_sum, per_n = 0.0, 0
        val_loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=val_transform),
            batch_size  = BATCH_SIZE,
            shuffle     = False,
            num_workers = 0,
            pin_memory  = False
        )
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True).unsqueeze(1)
                per_loss_sum += criterion(self.model(images), labels).item() * len(labels)
                per_n        += len(labels)
        personalised_loss = per_loss_sum / per_n

        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss'             : float(avg_loss),
             'personalised_loss': float(personalised_loss),
             'client_name'      : self.name}
        )

    def evaluate(self, parameters: NDArrays, config: Dict) -> Tuple[float, int, Dict]:
        self.set_parameters(parameters)
        m = evaluate_client(self.model, self.val_dataframe, self.device)
        m['client_name'] = self.name
        return float(1.0 - (m['auc'] if not np.isnan(m['auc']) else 0.5)), \
               len(self.val_dataframe), m

print("✓ GIFAIRPerHospitalClient defined — Yue et al. 2022 Algorithm 3")
print("  fit()      → trains on train_clients, computes personalised_loss")
print("  evaluate() → validates on val_clients after each round")

✓ GIFAIRPerHospitalClient defined — Yue et al. 2022 Algorithm 3
  fit()      → trains on train_clients, computes personalised_loss
  evaluate() → validates on val_clients after each round


In [ ]:
# GIFAIR-FL-Per strategy — Yue et al. 2022 Algorithm 3
#
# Key differences from GIFAIR-Global:
# 1. Server stores personalised weights theta_k for each hospital
# 2. r_k computed from personalised losses, not global model losses
# 3. After aggregation, server updates theta_k = local weights returned by client
# 4. Evaluation uses personalised theta_k per hospital

class GIFAIRPerStrategy(Strategy):

    def __init__(self, lam: float, num_clients: int, client_sizes: Dict[str, int]):
        super().__init__()
        self.lam         = lam
        self.num_clients = num_clients
        total    = sum(client_sizes.values())
        self.p_k = {name: size / total for name, size in client_sizes.items()}

        # Personalised losses — updated after each round from client's own model
        self.personalised_losses: Dict[str, float] = {n: 1.0 for n in client_sizes}

        # Personalised weights — theta_k for each hospital [Algorithm 3]
        self.personalised_params: Dict[str, Optional[NDArrays]] = \
            {n: None for n in client_sizes}

        self._global_params: Optional[NDArrays] = None

    def initialize_parameters(self, client_manager):
        ndarrays            = [v.cpu().numpy() for v in build_model().state_dict().values()]
        self._global_params = ndarrays
        # Initialise personalised params to global model
        for name in self.personalised_params:
            self.personalised_params[name] = [p.copy() for p in ndarrays]
        return ndarrays_to_parameters(ndarrays)

    def _compute_scales(self) -> Dict[str, float]:
        # r_k computed from PERSONALISED losses [Yue et al. 2022 Eq.5]
        # FIX: same lambda_max bound as GIFAIR-Global applies here
        # (Hospital_A safe; B, C, D, E violate lambda < p_k/(d-1)).
        # Floor prevents negative scale (=gradient ascent).
        names  = list(self.personalised_losses.keys())
        losses = [self.personalised_losses[n] for n in names]
        scales = {}
        for i, name in enumerate(names):
            r_k = sum(np.sign(losses[i] - losses[j])
                    for j in range(len(losses)) if j != i)
            p_k       = self.p_k[name]
            raw_scale = 1.0 + (self.lam / p_k) * r_k

            if not np.isfinite(raw_scale):
                print(f"  [WARNING] {name}: raw_scale non-finite ({raw_scale}) — using safe default 1.0")
                scales[name] = 1.0
            else:
                scales[name] = float(np.clip(raw_scale, 0.05, 5.0))
        return scales

    def configure_fit(self, server_round, parameters, client_manager):
        self._global_params = parameters_to_ndarrays(parameters)
        scales  = self._compute_scales()
        sampled = client_manager.sample(
            num_clients=self.num_clients,
            min_num_clients=self.num_clients
        )
        fit_ins_list = []
        for proxy in sampled:
            cid_int = int(proxy.cid) % NUM_CLIENTS
            name    = HOSPITAL_NAMES[cid_int]
            config  = {'epochs': 3, 'lr': 1e-4,
                       'scale': float(scales.get(name, 1.0)),
                       'server_round': server_round}
            fit_ins_list.append((proxy, FitIns(parameters, config)))
        return fit_ins_list

    def aggregate_fit(self, server_round, results, failures):
        if not results:
            return None, {}

        # Aggregation = plain unweighted average (same as GIFAIR-Global)
        all_weights = [parameters_to_ndarrays(r.parameters) for _, r in results]
        n     = len(all_weights)
        w_new = [
            sum(w[layer_idx] for w in all_weights) / n
            for layer_idx in range(len(all_weights[0]))
        ]

        # Update personalised weights and personalised losses [Algorithm 3]
        # Set theta_k = theta_k^{(c+1)E}  — client keeps its own trained weights
        for _, fit_res in results:
            cname = fit_res.metrics.get('client_name', '')
            if cname in self.personalised_params:
                # Store this client's personalised weights
                self.personalised_params[cname] = \
                    parameters_to_ndarrays(fit_res.parameters)
                # Update personalised loss for next round r_k computation
                self.personalised_losses[cname] = float(
                    fit_res.metrics.get('personalised_loss', 1.0))

        self._global_params = w_new
        return ndarrays_to_parameters(w_new), {}

    def configure_evaluate(self, server_round, parameters, client_manager):
        ins     = EvaluateIns(parameters, {})
        sampled = client_manager.sample(
            num_clients=self.num_clients,
            min_num_clients=self.num_clients
        )
        return [(client, ins) for client in sampled]

    def aggregate_evaluate(self, server_round, results, failures):
        if not results:
            return None, {}

        print(f"\n── Round {server_round}/{ROUNDS} Validation ──")
        for _, res in results:
            name = res.metrics.get('client_name', '?')
            auc  = res.metrics.get('auc', float('nan'))
            fnr  = res.metrics.get('fnr', float('nan'))
            print(f"  {name:<14} AUC: {auc:.4f}  FNR: {fnr:.4f}")
        aucs       = [r.metrics.get('auc', float('nan')) for _, r in results]
        valid_aucs = [a for a in aucs if not (a != a)]
        if valid_aucs:
            print(f"  Mean AUC : {sum(valid_aucs)/len(valid_aucs):.4f} | "
                  f"Variance : {float(np.var(valid_aucs)):.6f}")

        total_loss    = sum(r.loss * r.num_examples for _, r in results)
        total_samples = sum(r.num_examples for _, r in results)
        return total_loss / total_samples, {}

    def evaluate(self, server_round, parameters):
        return None

print("✓ GIFAIRPerStrategy defined — Yue et al. 2022 Algorithm 3")
print(f"  λ = {LAM}")
print("  r_k from PERSONALISED losses — key difference from GIFAIR-Global")
print("  theta_k stored per hospital after each round")

✓ GIFAIRPerStrategy defined — Yue et al. 2022 Algorithm 3
  λ = 0.1
  r_k from PERSONALISED losses — key difference from GIFAIR-Global
  theta_k stored per hospital after each round


In [ ]:
# GIFAIR-Per client factory

def make_gifair_per_client_fn(train_data_map, val_data_map, device):
    def client_fn(cid: str) -> fl.client.Client:
        name = HOSPITAL_NAMES[int(cid)]
        return GIFAIRPerHospitalClient(
            name          = name,
            dataframe     = train_data_map[name],
            val_dataframe = val_data_map[name],
            device        = device
        ).to_client()
    return client_fn

print(f"✓ GIFAIR-Per client factory defined")

✓ GIFAIR-Per client factory defined


In [ ]:
# Run GIFAIR-FL-Per simulation

import os, logging
os.environ['RAY_SILENT_MODE'] = '1'
logging.getLogger('flwr').setLevel(logging.ERROR)

print("=" * 60)
print("GIFAIR-FL-Per — Yue et al. 2022 Algorithm 3")
print(f"Rounds: {ROUNDS} | λ: {LAM} | Clients: {NUM_CLIENTS} | Split: 85/10/5 | Batch: {BATCH_SIZE}")
print("=" * 60)

t0 = time.time()

client_sizes        = {name: len(data) for name, data in train_clients.items()}
gifair_per_strategy = GIFAIRPerStrategy(
    lam          = LAM,
    num_clients  = NUM_CLIENTS,
    client_sizes = client_sizes
)

gifair_per_history = fl.simulation.start_simulation(
    client_fn        = make_gifair_per_client_fn(train_clients, val_clients, device),
    num_clients      = NUM_CLIENTS,
    config           = fl.server.ServerConfig(num_rounds=ROUNDS),
    strategy         = gifair_per_strategy,
    client_resources = {'num_gpus': 1.0}
)

print(f"\n{'='*60}")
print(f"✓ GIFAIR-FL-Per complete in {(time.time()-t0)/60:.1f} minutes")
print(f"\nLoss per round:")
for rnd, loss in gifair_per_history.losses_distributed:
    print(f"  Round {rnd:>2} : {loss:.4f}")

GIFAIR-FL-Per — Yue et al. 2022 Algorithm 3
Rounds: 10 | λ: 0.1 | Clients: 5 | Split: 85/10/5 | Batch: 512


2026-07-24 21:02:02,277	INFO worker.py:2012 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
(ClientAppActor pid=13999) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=13999) 
(ClientAppActor pid=13999)             This is a deprecated feature. It will be removed
(ClientAppActor pid=13999)             entirely in future versions of Flower.
(ClientAppActor pid=13999)         
(ClientAppActor pid=13999) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning


── Round 1/10 Validation ──
  Hospital_D     AUC: 0.6871  FNR: 0.1635
  Hospital_A     AUC: 0.7147  FNR: 0.1625
  Hospital_C     AUC: 0.7374  FNR: 0.1577
  Hospital_E     AUC: 0.6403  FNR: 0.2500
  Hospital_B     AUC: 0.8305  FNR: 0.1636
  Mean AUC : 0.7220 | Variance : 0.003991


(ClientAppActor pid=13999) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=13999) 
(ClientAppActor pid=13999)             This is a deprecated feature. It will be removed
(ClientAppActor pid=13999)             entirely in future versions of Flower.
(ClientAppActor pid=13999)         
(ClientAppActor pid=13999) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=13999) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=13999)   self.pid = os.fork()
(ClientAppActor pid=13999) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr


── Round 2/10 Validation ──
  Hospital_A     AUC: 0.7363  FNR: 0.1299
  Hospital_B     AUC: 0.7816  FNR: 0.1187
  Hospital_D     AUC: 0.6745  FNR: 0.1250
  Hospital_E     AUC: 0.6850  FNR: 0.2000
  Hospital_C     AUC: 0.7650  FNR: 0.1254
  Mean AUC : 0.7285 | Variance : 0.001804


...


── Round 8/10 Validation ──
  Hospital_D     AUC: 0.6728  FNR: 0.2019
  Hospital_B     AUC: 0.7900  FNR: 0.1692
  Hospital_C     AUC: 0.7372  FNR: 0.1971
  Hospital_E     AUC: 0.6825  FNR: 0.1750
  Hospital_A     AUC: 0.7361  FNR: 0.1726
  Mean AUC : 0.7237 | Variance : 0.001804


(ClientAppActor pid=13999) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=13999) 
(ClientAppActor pid=13999)             This is a deprecated feature. It will be removed
(ClientAppActor pid=13999)             entirely in future versions of Flower.
(ClientAppActor pid=13999)         
(ClientAppActor pid=13999) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=13999) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=13999)   self.pid = os.fork()
(ClientAppActor pid=13999) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr


── Round 9/10 Validation ──
  Hospital_E     AUC: 0.6599  FNR: 0.1500
  Hospital_A     AUC: 0.7049  FNR: 0.0991
  Hospital_C     AUC: 0.6972  FNR: 0.1183
  Hospital_D     AUC: 0.6428  FNR: 0.1058
  Hospital_B     AUC: 0.6645  FNR: 0.0916
  Mean AUC : 0.6739 | Variance : 0.000551


(ClientAppActor pid=13999) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=13999) 
(ClientAppActor pid=13999)             This is a deprecated feature. It will be removed
(ClientAppActor pid=13999)             entirely in future versions of Flower.
(ClientAppActor pid=13999)         
(ClientAppActor pid=13999) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=13999) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=13999)   self.pid = os.fork()
(ClientAppActor pid=13999) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr


── Round 10/10 Validation ──
  Hospital_A     AUC: 0.7250  FNR: 0.0677
  Hospital_B     AUC: 0.7293  FNR: 0.0664
  Hospital_D     AUC: 0.6660  FNR: 0.0673
  Hospital_E     AUC: 0.6768  FNR: 0.1750
  Hospital_C     AUC: 0.7245  FNR: 0.0896
  Mean AUC : 0.7043 | Variance : 0.000737

✓ GIFAIR-FL-Per complete in 648.5 minutes

Loss per round:
  Round  1 : 0.2699
  Round  2 : 0.2530
  Round  3 : 0.2438
  Round  4 : 0.2472
  Round  5 : 0.2484
  Round  6 : 0.2552
  Round  7 : 0.2913
  Round  8 : 0.2609
  Round  9 : 0.3036
  Round 10 : 0.2770


In [ ]:
# Evaluate GIFAIR-FL-Per using PERSONALISED weights per hospital
# This is the key difference from GIFAIR-Global evaluation
# Each hospital is evaluated on its own theta_k, not the global model

import gc, ray
if ray.is_initialized():
    ray.shutdown()
torch.cuda.empty_cache()
gc.collect()

gifair_per_metrics = {}

for name, data in test_clients.items():
    # Use personalised weights for this hospital [Algorithm 3 — Return {theta_k}]
    per_params = gifair_per_strategy.personalised_params[name]
    if per_params is None:
        # Fallback to global if personalised not available
        per_params = gifair_per_strategy._global_params

    model = build_model().to(device)
    model.load_state_dict(dict(zip(
        model.state_dict().keys(),
        [torch.tensor(p) for p in per_params]
    )))
    gifair_per_metrics[name] = evaluate_client(model, data, device)
    del model
    torch.cuda.empty_cache()

rows = []
for name in HOSPITAL_NAMES:
    m = gifair_per_metrics[name]
    rows.append({
        'Client'   : name,
        'AUC'      : m['auc'],
        'FNR'      : m['fnr'],
        'Accuracy' : m['accuracy'],
        'AUC_M'    : m.get('auc_M', 'N/A'),
        'FNR_M'    : m.get('fnr_M', 'N/A'),
        'AUC_F'    : m.get('auc_F', 'N/A'),
        'FNR_F'    : m.get('fnr_F', 'N/A'),
    })

df_gifair_per = pd.DataFrame(rows).set_index('Client')
print("=== GIFAIR-FL-Per Results — Yue et al. 2022 Algorithm 3 ===\n")
print(df_gifair_per.to_string())

valid_aucs = [gifair_per_metrics[n]['auc'] for n in HOSPITAL_NAMES
              if not (gifair_per_metrics[n]['auc'] != gifair_per_metrics[n]['auc'])]
auc_var = np.var(valid_aucs) if valid_aucs else float('nan')
print(f"\nAUC Variance (valid hospitals only) : {auc_var:.6f}")
print(f"Valid hospitals : {len(valid_aucs)}/5")
print("Note: evaluated on personalised theta_k per hospital, not global model")

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=3752) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=3752) is multi-threaded, use of fork() may lead to deadlocks in the child.
 

=== GIFAIR-FL-Per Results — Yue et al. 2022 Algorithm 3 ===

               AUC     FNR  Accuracy   AUC_M   FNR_M   AUC_F   FNR_F
Client                                                              
Hospital_A  0.7234  0.1493     67.35  0.7316  0.1431  0.7131  0.1578
Hospital_B     NaN  0.0000    100.00     NaN  0.0000     NaN  0.0000
Hospital_C  0.6519  0.8963     88.90  0.6979  0.9286  0.6019  0.8615
Hospital_D  0.6267  0.0385     72.97  0.8121  0.0303  0.5088  0.0526
Hospital_E  0.7298  0.7895     87.50  0.7456  0.8000  0.7081  0.7778

AUC Variance (valid hospitals only) : 0.001990
Valid hospitals : 4/5
Note: evaluated on personalised theta_k per hospital, not global model


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Save GIFAIR-FL-Per results to Drive

SAVE_DIR = '/content/drive/MyDrive/dissertation/results'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f'{SAVE_DIR}/gifair_per_full_metrics.json', 'w') as f:
    json.dump(gifair_per_metrics, f, indent=2, default=str)

# Save global model
torch.save(
    dict(zip(build_model().state_dict().keys(),
             [torch.tensor(v) for v in gifair_per_strategy._global_params])),
    f'{SAVE_DIR}/gifair_per_full_global_model.pth'
)

# Save personalised models per hospital
for name in HOSPITAL_NAMES:
    per_params = gifair_per_strategy.personalised_params[name]
    if per_params is not None:
        torch.save(
            dict(zip(build_model().state_dict().keys(),
                     [torch.tensor(v) for v in per_params])),
            f'{SAVE_DIR}/gifair_per_full_{name}_model.pth'
        )

print("✓ GIFAIR-Per metrics saved → gifair_per_full_metrics.json")
print("✓ GIFAIR-Per global model  → gifair_per_full_global_model.pth")
print("✓ GIFAIR-Per personalised models saved per hospital")

✓ GIFAIR-Per metrics saved → gifair_per_full_metrics.json
✓ GIFAIR-Per global model  → gifair_per_full_global_model.pth
✓ GIFAIR-Per personalised models saved per hospital


In [ ]:
import os

expected_files = [
    f'{SAVE_DIR}/gifair_per_full_metrics.json',
    f'{SAVE_DIR}/gifair_per_full_global_model.pth',
] + [f'{SAVE_DIR}/gifair_per_full_{name}_model.pth' for name in HOSPITAL_NAMES]

all_saved = all(os.path.exists(f) for f in expected_files)

if all_saved:
    print("✓ All 7 result files confirmed saved — disconnecting session")
    from google.colab import runtime
    runtime.unassign()
else:
    missing = [f for f in expected_files if not os.path.exists(f)]
    print("⚠ Missing files — NOT disconnecting:", missing)

✓ All 7 result files confirmed saved — disconnecting session
